# Wildcards & Window functions 

## Wildcards

- %   matches any sequence of characters (including none)
- _   matches exactly one character
- [a-z] matches any single character in a range (Databricks uses RLIKE for full regex)
- LIKE is case-insensitive in most databases; NOT LIKE excludes matching rows

| OVER() Clause                       | What it Means                     | Use Case                        |
| ----------------------------------- | --------------------------------- | ------------------------------- |
| **OVER()**                          | Whole table = one window          | Grand totals, overall averages  |
| **OVER(PARTITION BY x)**            | One window per group, no ordering | Group totals, group averages    |
| **OVER(ORDER BY y)**                | Whole table, ordered              | Running totals across all data  |
| **OVER(PARTITION BY x ORDER BY y)** | One ordered window per group      | Running totals per group, ranks |


In [0]:
select *
from brightlearn.brightcoffee.shop_sales
;

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

### Question 1
Find all products where the product_detail starts with the word 'Dark'.
Write a SELECT query using LIKE that returns transaction_id, product_detail, and unit_price.

In [0]:
SELECT  transaction_id,  product_detail,  unit_price
FROM    brightlearn.brightcoffee.shop_sales
WHERE   product_detail like 'Dark%'
LIMIT   10;

### Question 2 
Find all transactions where product_detail ends with 'Lg' (large size).
Return transaction_id, product_detail, product_category, and unit_price. Order by unit_price descending.

In [0]:
SELECT  transaction_id,  product_detail,  product_category,  unit_price
FROM    brightlearn.brightcoffee.shop_sales
WHERE   product_detail iLIKE '%lg' --ilike is not case sensitive
ORDER BY unit_price desc;

Databricks visualization. Run in Databricks to view.

In [0]:
select 
product_category,
sum(transaction_qty*unit_price) Total_revenue,
count(distinct transaction_id) Total_transactions
from brightlearn.brightcoffee.shop_sales
group by all;

Databricks visualization. Run in Databricks to view.

### Question 3
Find all products that contain the word 'Chai' anywhere in product_type.
Return product_type and product_detail. Use DISTINCT so each combination appears once.

In [0]:
SELECT  DISTINCT  product_type,  product_detail
FROM    brightlearn.brightcoffee.shop_sales
WHERE  product_type like '%Chai%'
ORDER BY  product_detail;

### Question 4
Use NOT LIKE to find all products that are NOT a type of tea.
Filter product_category so that it does NOT contain the word 'Tea'. Return DISTINCT product_category, product_type, and product_detail.

In [0]:
SELECT  DISTINCT  product_category,  product_type,  product_detail
FROM   brightlearn.brightcoffee.shop_sales
WHERE  product_category not ilike '%tea%'
ORDER BY  product_category,  product_type;

### Question 5
Use the underscore wildcard to find products with a 2-character size code at the end.
The size codes are 'Sm', 'Rg', and 'Lg'. These all follow a space and are exactly 2 characters. Match product_detail values that end with ' __' (space + any 2 chars). Return DISTINCT product_detail ordered alphabetically.


In [0]:
SELECT  DISTINCT  product_detail
FROM    brightlearn.brightcoffee.shop_sales
WHERE  product_detail like '% __'
ORDER BY  product_detail;

### Question 6
Use RLIKE (regex) to find all products where product_detail contains a number.
Return DISTINCT product_detail. (Hint: the regex pattern for 'contains a digit' is [0-9].)


In [0]:
SELECT  DISTINCT  product_detail
FROM    brightlearn.brightcoffee.shop_sales
WHERE   product_detail RLIKE '[0-9]'
ORDER BY  product_detail;

In [0]:
select distinct product_detail
from brightlearn.brightcoffee.shop_sales;


### Question 7
Combine two wildcard conditions in one query.
Find transactions where product_category is 'Coffee' AND product_detail contains either 'Brazilian' or 'Ethiopian' (tip: use two OR conditions with LIKE).
Return transaction_id, transaction_date, store_location, product_detail, unit_price. Order by transaction_date.



In [0]:
SELECT  transaction_id,  transaction_date,  store_location,
        product_detail,  unit_price
FROM    brightlearn.brightcoffee.shop_sales
WHERE  product_category = 'Coffee'
AND  (product_detail like '%Brazilian%'
        OR  product_detail like '%Ethiopia%'
        )
ORDER BY  transaction_date;

### Question 8  [WRITTEN - no SQL needed]

Explain in your own words what the following pattern matches:  product_detail  LIKE  '%Scone'  Then write a different LIKE pattern that would match product names that contain the word 'Blend' anywhere in the middle of the name.

## Window functions

|Category|Functions| Key point|
|-|-|-|
|**Aggregate**| SUM, AVG, COUNT, MIN, MAX| Add OVER() to any aggregate to keep all rows|
|**Ranking** |ROW_NUMBER, RANK, DENSE_RANK,NTILE|Always need ORDER BY inside OVER()|
|**Value**| LAG, LEAD, FIRST_VALUE, LAST_VALUE |Access other rows without a self-join|

### Question 9 
Calculate total revenue per store, keeping all individual rows.
Revenue = transaction_qty * unit_price. Use SUM() OVER(PARTITION BY store_location) to add a column called store_total_revenue to each row. Return transaction_id, store_location, transaction_qty, unit_price, and the new column. LIMIT 20.



In [0]:
SELECT
    transaction_id,
    store_location,
    transaction_qty,
    unit_price,
    sum(transaction_qty*unit_price) over(partition by store_location)  AS  store_total_revenue
FROM    brightlearn.brightcoffee.shop_sales
limit 20

### Question 10
For each transaction, show what percentage of its store's total revenue that transaction represents.
Calculate: ROUND( (transaction_qty * unit_price) * 100.0 / SUM(transaction_qty * unit_price) OVER(PARTITION BY store_location), 2 ) as pct_of_store_revenue. Return transaction_id, store_location, revenue (calculated), and the percentage column. LIMIT 20.

In [0]:
SELECT
    transaction_id,
    store_location,
    round(transaction_qty*unit_price,2) as Revenue ,
    ROUND( (transaction_qty * unit_price) * 100.0 / SUM(transaction_qty * unit_price) OVER(PARTITION BY store_location)
    ,5) pct_of_store_revenue
FROM    brightlearn.brightcoffee.shop_sales
limit 20;

### Question 11 
Calculate a running total of revenue per store, ordered by transaction date and time.
Add ORDER BY transaction_date, transaction_time inside the OVER clause of a SUM() window. This gives a cumulative (running) total that increases with each transaction. Return transaction_id, store_location, transaction_date, transaction_time, revenue, running_total. LIMIT 30.

In [0]:
select store_location, sum(transaction_qty*unit_price)
from brightlearn.brightcoffee.shop_sales
group by all

In [0]:
select 
transaction_id, 
store_location, 
transaction_date, 
transaction_time, 
(transaction_qty*unit_price) as revenue, 
round(sum(transaction_qty*unit_price) over (
    partition by store_location
    ORDER BY transaction_date, transaction_time
),2) as running_total
from brightlearn.brightcoffee.shop_sales
limit 30

### Question 12
Rank every transaction within its store by revenue (highest first).
Use ROW_NUMBER() with PARTITION BY store_location and ORDER BY revenue DESC. Return transaction_id, store_location, revenue, and row_num. LIMIT 30.


In [0]:

SELECT
    transaction_id,
    store_location,
    ROUND(transaction_qty * unit_price, 2)  AS  revenue,
    ROW_NUMBER()  OVER(
        partition by store_location
        order by transaction_qty * unit_price desc
    )  AS  row_num
FROM    brightlearn.brightcoffee.shop_sales
ORDER BY  store_location,  row_num
LIMIT   30;

In [0]:
select *,(transaction_qty*unit_price) revenue
from brightlearn.brightcoffee.shop_sales
order by revenue desc

In [0]:
select store_location, max(transaction_qty*unit_price) revenue
from brightlearn.brightcoffee.shop_sales
group by all; 

### Question 13
Compare RANK, DENSE_RANK, and ROW_NUMBER side by side on the same data.
Rank products by unit_price within each product_category, highest price first. Include all three functions as separate columns. LIMIT 40. Look for ties (same unit_price in same category) and observe how each function handles them.

In [0]:

SELECT
    product_category,
    product_detail,
    unit_price,
    ROW_NUMBER()  OVER( PARTITION BY product_category ORDER BY unit_price desc)  AS row_num,
    RANK()        OVER( PARTITION BY product_category ORDER BY unit_price desc)  AS rank,
    DENSE_RANK()  OVER( PARTITION BY product_category ORDER BY unit_price desc)  AS dense_rank
FROM    brightlearn.brightcoffee.shop_sales
ORDER BY  product_category,  unit_price  DESC
LIMIT   40;

### Question 14
Use NTILE to divide transactions into 4 revenue quartiles within each store.
Calculate revenue = transaction_qty * unit_price. Then use NTILE(4) OVER(PARTITION BY store_location ORDER BY revenue ASC) to assign a quartile (1 = lowest, 4 = highest). Return transaction_id, store_location, revenue, quartile. LIMIT 30.

999 rows

4 quartiles

999/4 -> (999 + 1)/4 -> 1000/4 ->(4x) 250 -> 

q1 = 250

q2 = 250

q3 = 250

q4 = 249

In [0]:
SELECT
    transaction_id,
    store_location,
    ROUND(transaction_qty * unit_price, 2)  AS  revenue,
    ntile(4) over(partition by store_location
    order by transaction_qty* unit_price asc) AS  quartile
FROM    brightlearn.brightcoffee.shop_sales
ORDER BY  store_location,  revenue  ASC
LIMIT   30;

### Question 15
Use LAG to compare each transaction's revenue to the previous transaction in the same store.
Order by transaction_date, transaction_time. Use LAG(revenue, 1, 0) - the third argument 0 is the default value when there is no previous row. Return transaction_id, store_location, transaction_date, revenue, prev_revenue. LIMIT 20.

In [0]:
SELECT
    transaction_id,
    store_location,
    transaction_date,
    ROUND(transaction_qty * unit_price, 2)  AS  revenue,
    ROUND( LAG(transaction_qty*unit_price)  OVER( partition by store_location
    order by transaction_date, transaction_date asc
    ), 2)  AS  prev_revenue
FROM brightlearn.brightcoffee.shop_sales
LIMIT   20;

### Question 16
Use LEAD to show the next transaction's revenue alongside the current one.
Same partition and order as Q15. Use LEAD(revenue, 1). Return transaction_id, store_location, transaction_date, revenue, next_revenue. LIMIT 20.

In [0]:
SELECT
    transaction_id,
    store_location,
    transaction_date,
    ROUND(transaction_qty * unit_price, 2)  AS  revenue,
    ROUND( LEAD(transaction_qty*unit_price)  OVER( partition by store_location
    order by transaction_date, transaction_date asc
    ), 2)  AS  next_revenue
FROM brightlearn.brightcoffee.shop_sales
LIMIT   20;

### Question 17
Use FIRST_VALUE to show the cheapest product in each category alongside every row.
Use FIRST_VALUE(product_detail) OVER(PARTITION BY product_category ORDER BY unit_price ASC). Return product_category, product_detail, unit_price, and cheapest_in_category. Use DISTINCT to avoid too many duplicate rows. LIMIT 30.

In [0]:
SELECT  DISTINCT
    product_category,
    product_detail,
    unit_price,
    FIRST_VALUE( product_detail )  OVER( partition by product_category
    order by unit_price asc
    )  AS  cheapest_in_category
FROM brightlearn.brightcoffee.shop_sales
ORDER BY  product_category,  unit_price
LIMIT   30;

### Question 18  - Challenge


Part A (1 mark):  Filter the data to only include rows where product_detail ends in 'Lg' or 'Rg' (large or regular sizes). Use LIKE with an OR condition. Assign revenue as transaction_qty * unit_price.

Part B (1 mark):  On that filtered data, use RANK() OVER(PARTITION BY store_location ORDER BY revenue DESC) to rank each transaction within its store.

Part C (1 mark):  Also add the store's average revenue using AVG() OVER(PARTITION BY store_location) as store_avg_revenue.

Return: transaction_id, store_location, product_detail, revenue, store_rank, store_avg_revenue. LIMIT 40.


In [0]:
SELECT
    transaction_id,
    store_location,
    product_detail,
    ROUND(transaction_qty * unit_price, 2)   AS  revenue,
    RANK() OVER(PARTITION BY store_location ORDER BY transaction_qty * unit_price DESC) AS  store_rank,
    ROUND( avg(transaction_qty * unit_price)  over (partition by store_location), 2)  AS  store_avg_revenue
FROM    brightlearn.brightcoffee.shop_sales
WHERE   product_detail ilike '%LG' or product_detail ilike '%RG'
ORDER BY  store_location,  store_rank
LIMIT   1000;